In [1]:
import os
from groq import Groq
from dotenv import load_dotenv

# .env dosyasını yükle
load_dotenv()

# Groq client
client = Groq(
    api_key = os.getenv("GROQ_API_KEY")
)

# İlk istek
response = client.chat.completions.create(
    model    = "llama-3.3-70b-versatile",
    messages = [
        {"role": "user", "content": "Merhaba! Türkçe cevap ver."}
    ]
)

print(response.choices[0].message.content)

Merhaba! Size nasıl yardımcı olabilirim?


In [2]:
import pandas as pd

# Verimizi yükle
df = pd.read_csv('islemler.csv')

# Kategori analizi
ozet = df.groupby('kategori').agg(
    islem_sayisi = ('tutar', 'count'),
    toplam_tutar = ('tutar', 'sum'),
    ort_tutar    = ('tutar', 'mean')
).round(2)

# Analizi LLM'e gönder
analiz_metni = ozet.to_string()

response = client.chat.completions.create(
    model    = "llama-3.3-70b-versatile",
    messages = [
        {
            "role": "system",
            "content": "Sen bir finansal veri analistisin. Türkçe, kısa ve net yorum yap."
        },
        {
            "role": "user",
            "content": f"Bu finansal işlem verisini analiz et ve 3 madde halinde içgörü sun:\n\n{analiz_metni}"
        }
    ]
)

print(response.choices[0].message.content)

Finansal işlem verisini analiz ettikten sonra aşağıdaki 3 madde halinde içgörü sunulabilir:

1. **En yüksek toplam tutar**: Transfer kategorisinde toplam 24.200,0 tutarında işlem yapıldığı görülmektedir. Bu, diğer kategorilere göre en yüksek tutarlı işlemlerin transfer kategorisinde yoğunlaştığını zeigt.
2. **Ortalama işlem tutarı**: Ortalama işlem tutarı en yüksek olan kategori transfer kategorisidir. Her bir transfer işleminin 평균 8.066,67 tutarında olduğu görülmektedir.
3. **İçsel dağılım**: Alışveriş kategorisinde 4 işlem gerçekleştirilirken, fatura ve transfer kategorilerinde 3'er işlem yapıldığı görülmektedir. Bu, müşterilerin alışveriş işlemlerini daha sık gerçekleştirdiğini, ancak transfer işlemlerinin daha yüksek tutarlarda yapıldığını göstermektedir.


In [3]:
def finansal_analiz(df, soru):
    """Veri hakkında doğal dilde soru sor, LLM cevaplasın"""
    
    ozet = df.describe().to_string()
    kategoriler = df.groupby('kategori')['tutar'].sum().to_string()
    durumlar = df.groupby('durum')['tutar'].sum().to_string()
    
    response = client.chat.completions.create(
        model    = "llama-3.3-70b-versatile",
        messages = [
            {
                "role": "system",
                "content": """Sen bir FinTech veri analistisin. 
                Sana finansal işlem verisi verilecek.
                Soruları Türkçe, kısa ve net cevapla.
                Sayısal verileri kullanarak cevapla."""
            },
            {
                "role": "user",
                "content": f"""
Veri özeti:
{ozet}

Kategori bazında tutarlar:
{kategoriler}

Durum bazında tutarlar:
{durumlar}

Soru: {soru}
"""
            }
        ]
    )
    return response.choices[0].message.content

# Test et
print(finansal_analiz(df, "Hangi kategori en riskli ve neden?"))

Transfer kategorisi en riskli görünüyor çünkü toplamda en yüksek tutar bu kategori altında yer alıyor (24.200). Bu, büyük miktarda paranın transfer edildiği anlamına geliyor, ki bu durum potansiyel olarak yüksek riskli olabilir.


In [4]:
sorular = [
    "İptal edilen işlemlerin ortak özellikleri nelerdir?",
    "Hangi müşteri en dikkat çekici davranış sergiliyor?",
    "Bu veriye göre fraud riski var mı?"
]

for soru in sorular:
    print(f"\n❓ {soru}")
    print("─" * 50)
    print(finansal_analiz(df, soru))
    print()


❓ İptal edilen işlemlerin ortak özellikleri nelerdir?
──────────────────────────────────────────────────
İptal edilen işlemlerin ortak özellikleri:
- Toplam tutar: 12.000,0
- Durum: İptal
Bu işlemler, yüksek tutarlı (12.000,0) işlemlerdir ve genel olarak yüksek tutarlı işlemler iptal edilmektedir.


❓ Hangi müşteri en dikkat çekici davranış sergiliyor?
──────────────────────────────────────────────────
Veri özeti ve kategori/durum bazında tutarlar dikkate alındığında, dikkat çekici davranış gösteren müşteri, en yüksek işlemleri gerçekleştiren müşteri olabilir. 

Müşteri_id'nin dağılımına bakıldığında, mean değeri 2.5, std değeri 1.27 ve max değeri 4.0'dir. Bu, müşteri_id'nin genellikle düşük değerlerde olduğunu, ancak bazı müşterilerin daha yüksek değerlerde işlem yaptığını gösterir.

Örneğin, 4.0 olan maksimum müşteri_id, diğer müşterilerden daha fazla işlem gerçekleştirmiş olabilir ve bu nedenle daha dikkat çekici davranmıştır.


❓ Bu veriye göre fraud riski var mı?
────────────────

## Gün 2 — LangChain Temelleri

In [7]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

# LangChain ile Groq bağlantısı
llm = ChatGroq(
    api_key     = os.getenv("GROQ_API_KEY"),
    model       = "llama-3.3-70b-versatile",
    temperature = 0
)

# Test
response = llm.invoke("Merhaba, Türkçe cevap ver.")
print(response.content)


Merhaba, size nasıl yardımcı olabilirim?


In [8]:
# Prompt şablonu
prompt = ChatPromptTemplate.from_messages([
    ("system", "Sen bir FinTech veri analistisin. Türkçe, kısa ve net cevap ver."),
    ("user", "Şu finansal veriyi analiz et:\n{veri}\n\nSoru: {soru}")
])

# Zincir oluştur — prompt + llm
zincir = prompt | llm

# Çalıştır
ozet = df.groupby('kategori')['tutar'].sum().to_string()

response = zincir.invoke({
    "veri": ozet,
    "soru": "Hangi kategori en fazla gelir getiriyor?"
})

print(response.content)

Transfer kategorisi en fazla geliri getiriyor, toplam 24200.0 tutarında.


In [9]:
from langchain_core.output_parsers import StrOutputParser

# Output parser ekle
parser = StrOutputParser()

# Yeni zincir: prompt → llm → parser
zincir2 = prompt | llm | parser

# Birden fazla soru sor
sorular = [
    "En yüksek riskli kategori hangisi?",
    "Onaylı işlemlerin toplam tutarı nedir?",
    "Bu veriden 1 cümleyle ne sonuç çıkarırsın?"
]

for soru in sorular:
    cevap = zincir2.invoke({
        "veri": ozet,
        "soru": soru
    })
    print(f"❓ {soru}")
    print(f"💬 {cevap}")
    print()

❓ En yüksek riskli kategori hangisi?
💬 En yüksek riskli kategori "Transfer" olarak görünüyor, çünkü en yüksek tutar bu kategoride.

❓ Onaylı işlemlerin toplam tutarı nedir?
💬 Onaylı işlemlerin toplam tutarı: 4765 + 1350 + 24200 = 30315.0

❓ Bu veriden 1 cümleyle ne sonuç çıkarırsın?
💬 Bu finansal veriye göre, transfer işlemleri toplam harcamaların büyük çoğunluğunu oluşturuyor.



In [10]:
from langchain_core.messages import HumanMessage, AIMessage

# Sohbet geçmişi
gecmis = []

def sohbet(soru):
    # Soruyu geçmişe ekle
    gecmis.append(HumanMessage(content=soru))
    
    # LLM'e gönder
    response = llm.invoke(
        [
            ("system", f"Sen bir FinTech analistisin. Bu veriyi kullan:\n{ozet}"),
        ] + gecmis
    )
    
    # Cevabı geçmişe ekle
    gecmis.append(AIMessage(content=response.content))
    
    return response.content

# Test — birbirine bağlı sorular
print(sohbet("En yüksek tutarlı kategori hangisi?"))
print()
print(sohbet("Peki bu kategoriyi kim kullanıyor olabilir?"))
print()
print(sohbet("Bu kişi için risk profili çıkar"))


Verilere göre en yüksek tutarlı kategori "Transfer"dir. Transfer kategorisinin tutarı 24.200,0 olarak görünüyor. Bu, diğer iki kategori olan "Alışveriş" (4.765,0) ve "Fatura" (1.350,0)'dan daha yüksek.

Bu kategorinin en yüksek tutarlı olması, büyük olasılıkla iş veya kurumsal müşterilerin kullanımını gösteriyor olabilir. Transfer kategorisinin tutarı 24.200,0 olarak görünüyor, bu da büyük miktarda para transferi yapıldığını gösteriyor.

Bunun beberapa nedeni olabilir:

* İşletmeler veya şirketler arasında büyük miktarda para transferi yapılması
* Kurumsal müşterilerin finansal işlemleri için yüksek tutarlı transferler yapılması
* Büyük ölçekli ticaret veya yatırım faaliyetleri için transferlerin yapılması

Ancak, daha spesifik bir cevap verebilmek için daha fazla veri ve bilgiye ihtiyaç duyulabilir.

Bu kişi için risk profili çıkarırken, aşağıdaki faktörleri dikkate alacağım:

* Transfer kategorisinin en yüksek tutarlı olması
* Büyük miktarda para transferi yapılması

Bu faktörler dik

## Gün 3 — RAG: Retrieval Augmented Generation

In [11]:
# Örnek finansal belgeler
belgeler = [
    """
    2024 Q1 Finansal Raporu:
    Şirketin toplam geliri 2.5 milyon TL olarak gerçekleşti.
    Transfer işlemleri toplam gelirin %65'ini oluşturdu.
    Fraud kayıpları 45.000 TL olarak tespit edildi.
    En riskli bölge İstanbul olarak belirlendi.
    """,
    """
    2024 Q2 Finansal Raporu:
    Toplam işlem sayısı 15.432 olarak kaydedildi.
    Ortalama işlem tutarı 3.200 TL'ye yükseldi.
    Alışveriş kategorisinde %23 büyüme gözlemlendi.
    Müşteri memnuniyeti %87 olarak ölçüldü.
    """,
    """
    Risk Analizi Raporu:
    Yüksek tutarlı transferlerde fraud riski %2.3 olarak hesaplandı.
    5000 TL üzeri işlemlerin %8'i şüpheli olarak işaretlendi.
    Gece saatlerinde yapılan işlemler daha yüksek risk taşıyor.
    KYC doğrulaması tamamlanmamış 234 hesap tespit edildi.
    """,
    """
    Müşteri Segmentasyonu:
    Premium müşteriler toplam işlemlerin %45'ini gerçekleştiriyor.
    Kurumsal müşterilerin ortalama işlem tutarı 12.500 TL.
    Bireysel müşterilerin ortalama işlem tutarı 1.800 TL.
    18-25 yaş segmentinde mobil işlemler %92 oranında.
    """
]

print(f"{len(belgeler)} belge hazır!")

4 belge hazır!


In [12]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

# Belgeleri Document formatına çevir
docs = [Document(page_content=belge) for belge in belgeler]

# Embedding modeli — ücretsiz, yerel çalışır
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# ChromaDB'ye kaydet
vectorstore = Chroma.from_documents(
    documents  = docs,
    embedding  = embeddings
)

print("Vektör veritabanı hazır!")

C:\Users\MSI\AppData\Local\Temp\ipykernel_26084\710690187.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


ModuleNotFoundError: No module named 'langchain.text_splitter'

In [13]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Önce kur
import subprocess
subprocess.run(["py", "-m", "pip", "install", "langchain-text-splitters", "langchain-huggingface"])

# Belgeleri Document formatına çevir
docs = [Document(page_content=belge) for belge in belgeler]

# Embedding modeli
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# ChromaDB'ye kaydet
vectorstore = Chroma.from_documents(
    documents = docs,
    embedding = embeddings
)

print("Vektör veritabanı hazır!")

C:\Users\MSI\AppData\Local\Temp\ipykernel_26084\3350371759.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\MSI\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vektör veritabanı hazır!


In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Retriever — en alakalı 2 belgeyi getir
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """Sen bir FinTech analistisin.
    Aşağıdaki belgeleri kullanarak soruyu Türkçe cevapla.
    Sadece belgelerdeki bilgileri kullan.
    
    Belgeler:
    {context}"""),
    ("user", "{soru}")
])

# Belgeleri birleştir
def belge_birlestir(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# RAG zinciri
rag_zinciri = (
    {"context": retriever | belge_birlestir, "soru": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test
cevap = rag_zinciri.invoke("Fraud kayıpları ne kadar?")
print(cevap)

Fraud kayıpları 45.000 TL olarak tespit edildi.


In [15]:
sorular = [
    "En riskli bölge neresi?",
    "Kurumsal müşterilerin ortalama işlem tutarı nedir?",
    "Q2'de kaç işlem yapıldı?",
    "Gece işlemleri hakkında ne biliyorsun?"
]

for soru in sorular:
    cevap = rag_zinciri.invoke(soru)
    print(f"❓ {soru}")
    print(f"💬 {cevap}")
    print()
    

❓ En riskli bölge neresi?
💬 En riskli bölge İstanbul olarak belirlendi.

❓ Kurumsal müşterilerin ortalama işlem tutarı nedir?
💬 Kurumsal müşterilerin ortalama işlem tutarı 12.500 TL'dir.

❓ Q2'de kaç işlem yapıldı?
💬 2024 Q2 Finansal Raporu'na göre, toplam 15.432 işlem yapıldı.

❓ Gece işlemleri hakkında ne biliyorsun?
💬 Gece saatlerinde yapılan işlemler daha yüksek risk taşıyor. Bu bilgi, Risk Analizi Raporu'nda yer alıyor.



In [16]:
import pandas as pd
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Fraud verisinden özet çıkar
fraud_df = pd.read_csv('creditcard.csv')

ozet = f"""
Toplam işlem: {len(fraud_df)}
Fraud işlem: {fraud_df['Class'].sum()}
Fraud oranı: {(fraud_df['Class'].sum()/len(fraud_df)*100).round(3)}%
Ortalama tutar: {fraud_df['Amount'].mean().round(2)}
Fraud ortalama tutar: {fraud_df[fraud_df['Class']==1]['Amount'].mean().round(2)}
Normal ortalama tutar: {fraud_df[fraud_df['Class']==0]['Amount'].mean().round(2)}
"""

print(ozet)


Toplam işlem: 284807
Fraud işlem: 492
Fraud oranı: 0.173%
Ortalama tutar: 88.35
Fraud ortalama tutar: 122.21
Normal ortalama tutar: 88.29



In [17]:
response = client.chat.completions.create(
    model = "llama-3.3-70b-versatile",
    messages = [
        {
            "role": "system",
            "content": "Sen bir FinTech fraud analisti sin. Türkçe, profesyonel ve kısa raporlar yaz."
        },
        {
            "role": "user",
            "content": f"""Bu fraud analizi verisinden profesyonel bir yönetici raporu yaz:

{ozet}

Rapor şunları içersin:
1. Genel durum özeti
2. Risk değerlendirmesi  
3. Öneriler"""
        }
    ]
)

print(response.choices[0].message.content)

**Yönetici Raporu: Fraud Analizi**

**1. Genel Durum Özeti**

Şirketimizdeki finansal işlemlerin analizi sonucunda, son dönemde toplam 284.807 işlem gerçekleştiği tespit edilmiştir. Bu işlemler içerisinde 492 adet şüpheli (fraud) işlem tespit edilmiştir. Fraud oranı %0,173 olarak hesaplanmış ve ortalama işlem tutarının 88,35 olduğu görülmüştür. Şüpheli işlemlerin ortalama tutarının 122,21, normal işlemlerin ortalama tutarının ise 88,29 olduğu belirlenmiştir.

**2. Risk Değerlendirmesi**

Fraud oranı %0,173 olarak hesaplanmış ve bu oran şirketimiz için önemli bir risk oluşturmaktadır. Şüpheli işlemlerin ortalama tutarının normal işlemlere kıyasla daha yüksek olması, bu işlemlerin daha büyük meblağlı ve daha kritik olduğunun göstergesi olabilir. Bu durum, şirketimiz için önemli bir finansal risk oluşturabilir ve önlemler alınmadığı takdirde daha büyük kayıplara neden olabilir.

**3. Öneriler**

1. **Güçlü Şifreleme ve Güvenlik Önlemleri**: Şirketimiz, finansal işlemlerin güvenliğini sağl